[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Реализуйте **training step с gradient accumulation**. Метод разбивает большой batch на несколько micro-batches: каждый помещается в память отдельно, но optimizer обновляет параметры только после накопления всех градиентов.

### Почему градиенты накапливаются
В PyTorch каждый вызов `loss.backward()` прибавляет новый градиент к `parameter.grad`, а не перезаписывает его. Поэтому `zero_grad()` нужно вызвать один раз перед всей группой micro-batches, а `optimizer.step()` — один раз после группы.

Если loss использует mean reduction и все micro-batches одинакового размера, деление каждого loss на число micro-batches даёт тот же средний градиент, что и один полный batch:

$$\nabla \bar L = \frac{1}{K} \sum_{k=1}^{K} \nabla L_k$$

Для micro-batches разного размера потребуется взвешивание по числу примеров; контракт этого упражнения предполагает обычный равный случай.

### Контракт
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### План реализации
1. Обнулите gradients один раз.
2. Для каждого `(x, y)` выполните forward и вычислите loss.
3. Масштабируйте loss на число micro-batches и вызовите backward.
4. После цикла выполните единственный optimizer step.
5. Верните средний loss как Python `float`, не используя его для backward после `.item()`.

### Быстрые самопроверки
- обновление совпадает с обновлением на склеенном полном batch;
- параметры действительно меняются;
- optimizer step вызывается ровно один раз;
- возвращаемое значение имеет тип `float`;
- при одном micro-batch алгоритм совпадает с обычным training step.

### Типичные ошибки
- `zero_grad()` внутри цикла: предыдущий micro-batch теряется;
- `optimizer.step()` внутри цикла: это уже несколько отдельных обновлений;
- отсутствие деления loss: эффективный gradient и learning rate становятся в `K` раз больше;
- вызов `.item()` до backward и попытка дифференцировать Python-число.

### Официальные материалы и примеры
- [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) — базовый цикл `zero_grad → backward → step`;
- [Zeroing out gradients](https://docs.pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html) — почему `.grad` накапливается;
- [`torch.autograd.backward`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.backward.html) — семантика накопления gradients.

### Вопросы для самопроверки
1. Почему zero_grad вызывается до группы, а не перед каждым micro-batch?
2. Зачем делить loss на число micro-batches?
3. Когда gradient accumulation не будет точно эквивалентен полному batch?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    pass  # zero_grad, loop (forward, scale loss, backward), step

In [ ]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')